# Global Beats: Tracking the Rise of Non-English Music (2018–2025)
### Data Analyst Portfolio Project

This notebook analyzes the "Global Crossover" effect in music, identifying trends in regional genre growth and listener loyalty.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set visual style
sns.set(style="whitegrid")
%matplotlib inline
print("Setup Complete")

In [ ]:
# Load data
df = pd.read_csv('spotify_data clean.csv')

# Clean dates
df['album_release_date'] = pd.to_datetime(df['album_release_date'], errors='coerce')
df['release_year'] = df['album_release_date'].dt.year
df = df.dropna(subset=['release_year', 'artist_genres'])
df['release_year'] = df['release_year'].astype(int)

print(f"Loaded {len(df)} records.")

In [ ]:
def get_detailed_category(genre_str):
    genre_str = genre_str.lower()
    mapping = {
        'Latino/Hispanic': ['latin', 'reggaeton', 'trap latino', 'mexicana', 'urbano', 'spanish'],
        'Asian (K/J/C-Pop)': ['k-pop', 'j-pop', 'v-pop', 'anime', 'korean', 'japanese'],
        'European (Non-Eng)': ['german', 'french', 'italian', 'portuguese', 'brazilian'],
        'Other Global': ['afrobeats', 'amapiano', 'desi', 'bollywood', 'punjabi', 'arabic', 'turkish']
    }
    for cat, keywords in mapping.items():
        if any(kw in genre_str for kw in keywords):
            return cat
    return 'English/Mainstream'

df['detailed_category'] = df['artist_genres'].apply(get_detailed_category)
print("Feature Engineering Complete.")

In [ ]:
recent_df = df[(df['release_year'] >= 2018) & (df['release_year'] <= 2025)]
trend_data = recent_df.groupby(['release_year', 'detailed_category']).size().unstack(fill_value=0)
trend_pct = trend_data.div(trend_data.sum(axis=1), axis=0) * 100

plt.figure(figsize=(10, 6))
global_cols = [c for c in trend_pct.columns if c != 'English/Mainstream']
for col in global_cols:
    plt.plot(trend_pct.index, trend_pct[col], marker='o', linewidth=2.5, label=col)

plt.title('Growth of Global Music Categories (Market Share %)')
plt.ylabel('Percentage of Tracks (%)')
plt.legend(title='Category', loc='upper left')
plt.show()

In [ ]:
genre_df = df.copy()
genre_df['genre_list'] = genre_df['artist_genres'].apply(lambda x: [g.strip() for g in x.split(',')] if isinstance(x, str) else [])
exploded = genre_df.explode('genre_list')
artist_stats = exploded.groupby(['genre_list', 'artist_name'])['artist_followers'].first().reset_index()
genre_loyalty = artist_stats.groupby('genre_list').agg({'artist_followers': 'mean', 'artist_name': 'count'})
top_loyalty = genre_loyalty[genre_loyalty['artist_name'] > 15].sort_values('artist_followers', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_loyalty, x='artist_followers', y=top_loyalty.index, palette='Blues_r')
plt.title('Top 10 Genres by Artist "Star Power"')
plt.show()